#### Setup

In [2]:
import pandas as pd

In [3]:
item_level = pd.read_parquet("data/processed/item_level.parquet")

#### Check for the number of unique products (n_unique_products) per order

In [4]:
n_products_per_order = item_level.groupby("order_id")["product_id"].nunique()
print(n_products_per_order.value_counts(normalize=True).sort_index() * 100)

product_id
1    96.720248
2     2.884479
3     0.302029
4     0.070946
5     0.008108
6     0.010135
7     0.003041
8     0.001014
Name: proportion, dtype: float64


In [5]:
more_than_one = (n_products_per_order >= 2).sum()
print(f'Number of orders which have more than one products: {more_than_one}')

Number of orders which have more than one products: 3236


**96.72% of orders contain only a single product**, with just 3.28% containing 2 or more distinct products (3,236 orders out of ~99,000). This is not a data quality issue — it reflects a genuine behavioral pattern of the platform.

This finding mirrors what was already observed in the RFM analysis, where 96.96% of customers made only one purchase. Together, these two patterns point to the same underlying characteristic of Olist as a marketplace: **customers tend to visit with a specific, single purchase intent**, rather than browsing and accumulating multiple items into a shopping cart the way they might on platforms like Amazon or Shopee. This is common for marketplaces aggregating many independent third-party sellers, where each seller effectively operates as its own small store rather than a unified catalog encouraging basket-building.

The practical consequence is that **traditional market basket analysis (Apriori/FP-Growth) will have very limited statistical power** when applied to the full order set — with such a small proportion of multi-item baskets, support for any product-pair association would be near zero across the ~99,000-order base. Rather than forcing an analysis that would yield noisy or meaningless rules, the more honest approach is to:
1. Report this as a standalone business insight about platform usage behavior, and
2. Narrow the market basket analysis to the subset of genuinely multi-item orders (3,236 orders), where any patterns found reflect real co-purchase behavior rather than statistical noise.

This also has a direct implication for cross-sell strategy: since bundling/cross-sell prompts at checkout are unlikely to be effective for a mostly single-item audience, growth efforts may be better directed toward **post-purchase or email-based cross-sell** (e.g., "customers who bought X also bought Y" recommendations sent after delivery) rather than in-cart upsell mechanics.

In [6]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

multi_item_orders = n_products_per_order[n_products_per_order >= 2].index
basket_df = item_level[item_level["order_id"].isin(multi_item_orders)]

# Use category instead of specific product_id — because product_id 
# is too detailed (tens of thousands of individual products)
# support will be almost = 0 for every pair. Category gives higher support,
# rules are easier to interpret.
transactions = (
    basket_df.groupby("order_id")["product_category_name_english"]
    .apply(lambda x: list(set(x)))
    .tolist()
)

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
basket_encoded = pd.DataFrame(te_array, columns=te.columns_)

frequent_itemsets = apriori(basket_encoded, min_support=0.01, use_colnames=True)
print(f"Number of itemset founded: {len(frequent_itemsets)}")

Number of itemset founded: 25


In [7]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
rules = rules.sort_values("lift", ascending=False)

rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(15)

,antecedents,consequents,support,confidence,lift
0,frozenset({home_confort}),frozenset({bed_bath_table}),0.013288,0.741379,2.998879
1,frozenset({bed_bath_table}),frozenset({home_confort}),0.013288,0.053750,2.998879


In [8]:
rules["support_count"] = (rules["support"] * len(transactions)).round().astype(int)
rules[["antecedents", "consequents", "support_count", "confidence", "lift"]].sort_values("lift", ascending=False).head(15)

,antecedents,consequents,support_count,confidence,lift
0,frozenset({home_confort}),frozenset({bed_bath_table}),43,0.741379,2.998879
1,frozenset({bed_bath_table}),frozenset({home_confort}),43,0.053750,2.998879


Among the 3,236 multi-item orders, only one statistically meaningful association was found: bed_bath_table and home_confort (lift = 3.0, based on 43 co-occurring orders). Customers who purchase from the home_confort category have a 74.1% chance of also purchasing a bed_bath_table item — nearly 3 times higher than random chance — suggesting these two categories serve a shared "home comfort/bedroom setup" need. However, the reverse is not true: bed_bath_table buyers only have a 5.4% chance of also buying home_confort, since bed_bath_table is a far more common category on its own.

Given that this is the only rule surfaced from the entire multi-item subset, and it's based on a modest sample of 43 orders, this should be treated as a directional signal rather than a robust pattern — worth testing (e.g., as a "customers also bought" prompt on home_confort product pages) rather than a confirmed strategy. Combined with the earlier finding that 96.7% of orders are single-item, this reinforces that cross-sell opportunities on Olist are narrow and category-specific, rather than a broad platform-wide pattern — bundling strategies should be considered only for a small number of well-validated category pairs like this one, not applied generally.

#### Recommended action
**1. A/B Testing:** suggests "customers who buy `home_confort` also often buy `bed_bath_table`" on the `home_confort` product page.

**2. Do not invest large resources in a large-scale bundle/cross-sell system:** Data shows that this is not a common behavioral pattern among Olist customers.

#### Saving outputs

In [9]:
# Save association rules table
rules_to_save = rules[["antecedents", "consequents", "support", "support_count", "confidence", "lift"]].copy()

# Convert frozenset to string since parquet doesnt support frozenset natively
rules_to_save["antecedents"] = rules_to_save["antecedents"].apply(lambda x: ", ".join(list(x)))
rules_to_save["consequents"] = rules_to_save["consequents"].apply(lambda x: ", ".join(list(x)))

rules_to_save.to_parquet("data/processed/market_basket_rules.parquet", index=False)
print(f"Saved {len(rules_to_save)} rules to data/processed/market_basket_rules.parquet")

Saved 2 rules to data/processed/market_basket_rules.parquet
